In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPool2D
from keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import splitfolders

splitfolders.ratio(r"C:\Users\Luis\anaconda3\fruits\fruits-360_dataset\fruits-360\Training",
                   output=r"C:\Users\Luis\anaconda3\output",
    seed=1337, ratio=(.7, .3), group_prefix=None, move=False) # default values


train_path =r"C:\Users\Luis\anaconda3\output\train"
validation_path =r"C:\Users\Luis\anaconda3\output\val"
test_path = r"C:\Users\Luis\anaconda3\fruits\fruits-360_dataset\fruits-360\Test"

batch_size= 32

train_datagen = ImageDataGenerator(rescale=1./255,
                      shear_range=0.3,
                      horizontal_flip=True,
                      zoom_range=0.3)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_Generator = train_datagen.flow_from_directory(train_path,
                                                          target_size=[100,100],
                                                          batch_size= batch_size,
                                                          color_mode="rgb",
                                                          class_mode="categorical",
                                                          shuffle=True)
val_Generator = val_datagen.flow_from_directory(validation_path,
                                                  target_size=[100,100],
                                                    batch_size=batch_size,
                                                    color_mode="rgb",
                                                    class_mode="categorical",
                                                    shuffle=False)
test_Generator = test_datagen.flow_from_directory(test_path,
                                                  target_size=[100,100],
                                                    batch_size=batch_size,

                                                  color_mode="rgb",
                                                  class_mode="categorical")

model = Sequential()
model.add(Conv2D(64,kernel_size=(3,3),input_shape=(100,100,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Conv2D(64,kernel_size=(3,3),activation="relu"))
model.add(MaxPool2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(1024, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(131,activation="softmax"))


model.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["accuracy"])

history = model.fit(
        train_Generator,
        steps_per_epoch=50,
        epochs=50,
        validation_data=val_Generator,
        validation_steps=15
    )

Copying files: 67692 files [01:37, 693.78 files/s]


Found 47345 images belonging to 131 classes.
Found 20347 images belonging to 131 classes.
Found 22688 images belonging to 131 classes.
Epoch 1/50
50/50 [==============================] - 15s 292ms/step - loss: 4.8633 - accuracy: 0.0188 - val_loss: 4.7263 - val_accuracy: 0.0000e+00
Epoch 2/50
50/50 [==============================] - 15s 290ms/step - loss: 3.9738 - accuracy: 0.0962 - val_loss: 3.6640 - val_accuracy: 0.0188
Epoch 3/50
50/50 [==============================] - 15s 290ms/step - loss: 3.1921 - accuracy: 0.2006 - val_loss: 2.3377 - val_accuracy: 0.3417
Epoch 4/50
50/50 [==============================] - 14s 286ms/step - loss: 2.3427 - accuracy: 0.3456 - val_loss: 1.8063 - val_accuracy: 0.4542
Epoch 5/50
50/50 [==============================] - 14s 289ms/step - loss: 1.9070 - accuracy: 0.4563 - val_loss: 1.7207 - val_accuracy: 0.5333
Epoch 6/50
50/50 [==============================] - 15s 294ms/step - loss: 1.6535 - accuracy: 0.5142 - val_loss: 1.2105 - val_accuracy: 0.6187
Epo

In [2]:
model.evaluate(test_Generator)

709/709 [==============================] - 43s 60ms/step - loss: 0.2552 - accuracy: 0.9218


[0.2552138864994049, 0.9218088984489441]

In [3]:
# Convert the model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model.
with open('model.tflite', 'wb') as f:
  f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\Luis\AppData\Local\Temp\tmpbuz2mkpz\assets


In [4]:
from tflite_support.metadata_writers import image_classifier
from tflite_support.metadata_writers import writer_utils

In [5]:
ImageClassifierWriter = image_classifier.MetadataWriter
_MODEL_PATH = r"C:\Users\Luis\anaconda3\model.tflite"
# Task Library expects label files that are in the same format as the one below.
_LABEL_FILE = r"C:\Users\Luis\anaconda3\labelfileallfruits.txt"
_SAVE_TO_PATH = r"C:\Users\Luis\anaconda3\allfruits.tflite"
# Normalization parameters is required when reprocessing the image. It is
# optional if the image pixel values are in range of [0, 255] and the input
# tensor is quantized to uint8. See the introduction for normalization and
# quantization parameters below for more details.
# https://www.tensorflow.org/lite/models/convert/metadata#normalization_and_quantization_parameters)
_INPUT_NORM_MEAN = 127.5
_INPUT_NORM_STD = 127.5

# Create the metadata writer.
writer = ImageClassifierWriter.create_for_inference(
    writer_utils.load_file(_MODEL_PATH), [_INPUT_NORM_MEAN], [_INPUT_NORM_STD],
    [_LABEL_FILE])

# Verify the metadata generated by metadata writer.
print(writer.get_metadata_json())

# Populate the metadata into the model.
writer_utils.save_file(writer.populate(), _SAVE_TO_PATH)

{
  "name": "ImageClassifier",
  "description": "Identify the most prominent object in the image from a known set of categories.",
  "subgraph_metadata": [
    {
      "input_tensor_metadata": [
        {
          "name": "image",
          "description": "Input image to be classified.",
          "content": {
            "content_properties_type": "ImageProperties",
            "content_properties": {
              "color_space": "RGB"
            }
          },
          "process_units": [
            {
              "options_type": "NormalizationOptions",
              "options": {
                "mean": [
                  127.5
                ],
                "std": [
                  127.5
                ]
              }
            }
          ],
          "stats": {
            "max": [
              1.0
            ],
            "min": [
              -1.0
            ]
          }
        }
      ],
      "output_tensor_metadata": [
        {
          "name": "proba